**1 - Set up Load Required Dependencies, Dataset and LLM**

**1.1 - Set up Required Dependencies**


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np

In [1]:
import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import evaluate
import peft

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("peft:", peft.__version__)

numpy: 1.26.4
pandas: 2.2.2
torch: 2.5.1+cu124
transformers: 4.38.2
datasets: 2.17.0
evaluate: 0.4.0
peft: 0.3.0


**Notes**

* The dependency installation completed successfully, although Colab showed compatibility warnings because some pre-installed packages expected different versions. These warnings are not blocking as long as the required lab imports run correctly.

* The environment was successfully configured after restarting the runtime. The required libraries for full fine-tuning, ROUGE evaluation, and PEFT/LoRA are now correctly available.


**1.2 - Load Dataset and LLM**


In [2]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np

huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/datasets/download/streaming_download_manager.py:784: FutureWarning: The 'verbose' keyword in pd.read_csv is deprecated and will be removed in a future version.
  return pd.read_csv(xopen(filepath_or_buffer, "rb", download_config=download_config), **kwargs)


Generating validation split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/datasets/download/streaming_download_manager.py:784: FutureWarning: The 'verbose' keyword in pd.read_csv is deprecated and will be removed in a future version.
  return pd.read_csv(xopen(filepath_or_buffer, "rb", download_config=download_config), **kwargs)


Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/datasets/download/streaming_download_manager.py:784: FutureWarning: The 'verbose' keyword in pd.read_csv is deprecated and will be removed in a future version.
  return pd.read_csv(xopen(filepath_or_buffer, "rb", download_config=download_config), **kwargs)


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [3]:
model_name = 'google/flan-t5-base'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

original_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16
).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

**Notes**

The DialogSum dataset and the original FLAN-T5 model were loaded successfully. The dataset provides dialogue-summary pairs for training and evaluation, while FLAN-T5 serves as the base sequence-to-sequence model that will later be compared against full fine-tuning and PEFT/LoRA approaches.

In [4]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 247577856
all model parameters: 247577856
percentage of trainable model parameters: 100.00%


**Notes**

The original FLAN-T5 model has 247,577,856 parameters, and 100% of them are trainable. This means that full fine-tuning would update the entire model, which can improve task performance but requires more compute and memory resources.


**1.3 — Test the Model with Zero Shot Inferencing**


In [5]:
index = 200

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

inputs = tokenizer(prompt, return_tensors='pt').to(device)

output = tokenizer.decode(
    original_model.generate(
        inputs["input_ids"],
        max_new_tokens=200,
    )[0],
    skip_special_tokens=True
)

dash_line = '-'.join('' for x in range(100))

print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Summarize the following conversation.

#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.

Summary:

-------------------------------------------------------------------

**Notes**

The original FLAN-T5 model can identify the general topic of the dialogue in zero-shot inference, but its summary is incomplete. It extracts a relevant sentence instead of generating a full human-like summary. This shows why fine-tuning can improve the model for the dialogue summarization task.

**2 - Perform Full Fine-Tuning**

**2.1 — Preprocess the Dialog-Summary Dataset**


In [6]:
def tokenize_function(example):
    start_prompt = 'Summarize the following conversation.\n\n'
    end_prompt = '\n\nSummary: '
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example["dialogue"]]

    example['input_ids'] = tokenizer(
        prompt,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).input_ids

    example['labels'] = tokenizer(
        example["summary"],
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).input_ids

    return example

tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns(
    ['id', 'topic', 'dialogue', 'summary']
)

tokenized_datasets = tokenized_datasets.filter(
    lambda example, index: index % 100 == 0,
    with_indices=True
)

print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

print(tokenized_datasets)

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Shapes of the datasets:
Training: (125, 2)
Validation: (5, 2)
Test: (15, 2)
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 125
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 15
    })
})


**Notes**

The DialogSum dataset was successfully preprocessed for fine-tuning. Each dialogue was converted into an instruction prompt and tokenized into input_ids, while each human summary was tokenized into labels. The dataset was then reduced to a smaller subset to make training faster during the lab.


In [ ]:
output_dir = f'./dialogue-summary-training-{str(int(time.time()))}'

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=1,
    max_steps=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    report_to="none",
    fp16=False,
    bf16=False,
    save_strategy="no"
)

trainer = Trainer(
    model=original_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation']
)

trainer.train()

**Notes**

The full fine-tuning pipeline was prepared, but running full training in the current Colab environment was too slow. This is expected because full fine-tuning updates all model parameters and requires more compute and memory resources.

In [1]:
!ls -al ./flan-dialogue-summary-checkpoint/

ls: cannot access './flan-dialogue-summary-checkpoint/': No such file or directory


In [3]:
from datasets import load_dataset, DatasetDict

huggingface_dataset_name = "knkarthick/dialogsum"
dataset = load_dataset(huggingface_dataset_name)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})


In [4]:
from datasets import DatasetDict
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq

demo_model_name = "google/flan-t5-small"

demo_tokenizer = AutoTokenizer.from_pretrained(demo_model_name)

raw_tiny_dataset = DatasetDict({
    "train": dataset["train"].shuffle(seed=42).select(range(24)),
    "validation": dataset["validation"].shuffle(seed=42).select(range(4)),
    "test": dataset["test"].shuffle(seed=42).select(range(4))
})

def tokenize_demo_function(example):
    start_prompt = "Summarize the following conversation.\n\n"
    end_prompt = "\n\nSummary:"

    prompts = [
        start_prompt + dialogue + end_prompt
        for dialogue in example["dialogue"]
    ]

    model_inputs = demo_tokenizer(
        prompts,
        max_length=256,
        truncation=True
    )

    labels = demo_tokenizer(
        text_target=example["summary"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_tiny_dataset = raw_tiny_dataset.map(
    tokenize_demo_function,
    batched=True,
    remove_columns=raw_tiny_dataset["train"].column_names
)

print(tokenized_tiny_dataset)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 24
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
})


**Notes**

The small DialogSum dataset was successfully preprocessed for the educational fine-tuning experiment. Dialogues were converted into instruction prompts and tokenized as input_ids, while summaries were tokenized as labels. The resulting dataset is ready for a lightweight training demonstration.

**2.2 — Load Demo Model and Generate Baseline Before Training**

In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [7]:
demo_model = AutoModelForSeq2SeqLM.from_pretrained(demo_model_name).to(device)

def generate_demo_summary(model, dialogue):
    prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""
    inputs = demo_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    output_ids = model.generate(
        inputs["input_ids"],
        max_new_tokens=80
    )

    return demo_tokenizer.decode(output_ids[0], skip_special_tokens=True)

test_dialogues = raw_tiny_dataset["test"]["dialogue"]
human_summaries = raw_tiny_dataset["test"]["summary"]

before_training_summaries = []

for dialogue in test_dialogues:
    before_training_summaries.append(generate_demo_summary(demo_model, dialogue))

print("Baseline generated summaries before training:")

for i, summary in enumerate(before_training_summaries):
    print(f"\nExample {i + 1}:")
    print(summary)

Baseline generated summaries before training:

Example 1:
Muslims.

Example 2:
You'd like to get the stamps.

Example 3:
Lily, what time would you like to go?

Example 4:
What are you going to order?


**Notes**

The small FLAN-T5 model generated baseline summaries before training. The outputs were incomplete and often copied short parts of the dialogue, which provides a useful baseline for comparing the effect of the mini fine-tuning step.

**Mini Fine-Tune the Model with the Preprocessed Dataset**

In [8]:
from transformers import TrainingArguments, Trainer
import time

data_collator = DataCollatorForSeq2Seq(
    tokenizer=demo_tokenizer,
    model=demo_model
)

output_dir = f'./demo-dialogue-summary-training-{str(int(time.time()))}'

demo_training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    max_steps=5,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

demo_trainer = Trainer(
    model=demo_model,
    args=demo_training_args,
    train_dataset=tokenized_tiny_dataset["train"],
    eval_dataset=tokenized_tiny_dataset["validation"],
    data_collator=data_collator
)

demo_trainer.train()

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Step,Training Loss
1,2.528500
2,1.935900
3,1.840300
4,1.589300
5,2.393900


TrainOutput(global_step=5, training_loss=2.057560849189758, metrics={'train_runtime': 13.3498, 'train_samples_per_second': 0.375, 'train_steps_per_second': 0.375, 'total_flos': 396469592064.0, 'train_loss': 2.057560849189758, 'epoch': 0.21})

**Notes**

The mini fine-tuning process ran successfully using FLAN-T5-small and a tiny subset of DialogSum. The model completed 5 training steps, confirming that the preprocessing, Trainer configuration, data collator, and training pipeline work correctly. This lightweight experiment demonstrates the full fine-tuning workflow without requiring the heavy compute resources needed for FLAN-T5-base.

**2.3 — Evaluate the Model Qualitatively**

In [9]:
after_training_summaries = []

for dialogue in test_dialogues:
    after_training_summaries.append(generate_demo_summary(demo_model, dialogue))

dash_line = "-".join("" for x in range(100))

for i in range(len(test_dialogues)):
    print(dash_line)
    print(f"Example {i + 1}")
    print(dash_line)
    print("HUMAN BASELINE SUMMARY:")
    print(human_summaries[i])
    print(dash_line)
    print("MODEL BEFORE MINI FINE-TUNING:")
    print(before_training_summaries[i])
    print(dash_line)
    print("MODEL AFTER MINI FINE-TUNING:")
    print(after_training_summaries[i])
    print()

---------------------------------------------------------------------------------------------------
Example 1
---------------------------------------------------------------------------------------------------
HUMAN BASELINE SUMMARY:
#Person1# and #Person2# talk about pilgrims around the world, including Muslims' pilgrimage to mecca and Christians' pilgrimage to Canterbury or Vatican. #Person2# thinks faith heals people instead of magical places.
---------------------------------------------------------------------------------------------------
MODEL BEFORE MINI FINE-TUNING:
Muslims.
---------------------------------------------------------------------------------------------------
MODEL AFTER MINI FINE-TUNING:
#Person1#: I am a pilgrim. #Person2#: I am a muslim. #Person1#: I am a Muslim. #Person2#: I am a muslim. #Person1#: #Person2#: I am a Muslim. #Person1#: #Person2

---------------------------------------------------------------------------------------------------
Example 2
------

**Notes**

The mini fine-tuning process changed the model outputs, which confirms that training affected the model. However, because the training used only a tiny dataset and 5 training steps, the generated summaries did not consistently improve. This shows that fine-tuning requires enough data, training steps and compute resources to produce meaningful improvements.

**2.4 — Evaluate the Model Quantitatively with ROUGE**


In [11]:
import evaluate
import numpy as np

In [12]:
rouge = evaluate.load("rouge")

before_results = rouge.compute(
    predictions=before_training_summaries,
    references=human_summaries,
    use_aggregator=True,
    use_stemmer=True
)

after_results = rouge.compute(
    predictions=after_training_summaries,
    references=human_summaries,
    use_aggregator=True,
    use_stemmer=True
)

print("BEFORE MINI FINE-TUNING:")
print(before_results)

print("\nAFTER MINI FINE-TUNING:")
print(after_results)

BEFORE MINI FINE-TUNING:
{'rouge1': 0.10436432637571157, 'rouge2': 0.0, 'rougeL': 0.10436432637571158, 'rougeLsum': 0.10436432637571158}

AFTER MINI FINE-TUNING:
{'rouge1': 0.2814872866597004, 'rouge2': 0.03125, 'rougeL': 0.24508881922675024, 'rougeLsum': 0.24508881922675022}


In [13]:
print("Absolute improvement after mini fine-tuning:")

improvement = (
    np.array(list(after_results.values())) -
    np.array(list(before_results.values()))
)

for key, value in zip(after_results.keys(), improvement):
    print(f"{key}: {value * 100:.2f}%")

Absolute improvement after mini fine-tuning:
rouge1: 17.71%
rouge2: 3.12%
rougeL: 14.07%
rougeLsum: 14.07%


**Conclusion Notes**

* The lightweight full fine-tuning experiment successfully demonstrated the complete fine-tuning workflow. The dataset was preprocessed into input_ids and labels, a small FLAN-T5 model was trained for 5 steps, and the model was evaluated qualitatively and quantitatively.

* The qualitative evaluation showed that the model outputs changed after training, although the summaries were still not consistently human-quality due to the tiny dataset and very limited number of training steps.

* The ROUGE evaluation showed measurable improvement after mini fine-tuning: rouge1 improved by 17.71%, rouge2 by 3.12%, rougeL by 14.07%, and rougeLsum by 14.07%. This confirms that fine-tuning can improve task alignment, but meaningful performance requires more training data, more steps, and stronger compute resources.

**3 - Perform Parameter Efficient Fine-Tuning (PEFT)**

**3.1 — Setup the PEFT/LoRA model for Fine-Tuning**


In [15]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0

    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()

    percentage = 100 * trainable_model_params / all_model_params

    return (
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}\n"
        f"percentage of trainable model parameters: {percentage:.2f}%"
    )

In [16]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

peft_demo_model = get_peft_model(
    demo_model,
    lora_config
)

print(print_number_of_trainable_model_parameters(peft_demo_model))

trainable model parameters: 1376256
all model parameters: 78337408
percentage of trainable model parameters: 1.76%


**Notes**

The PEFT/LoRA setup was successful. Instead of training all model parameters, LoRA reduced the trainable parameters to only 1.76% of the model. This demonstrates the main advantage of parameter-efficient fine-tuning: adapting a model to a specific task using far fewer compute and memory resources than full fine-tuning.

**3.2 — Train PEFT Adapter**

In [17]:
output_dir = f'./peft-demo-dialogue-summary-training-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    num_train_epochs=1,
    logging_steps=1,
    max_steps=5,
    save_strategy="no",
    report_to="none"
)

peft_trainer = Trainer(
    model=peft_demo_model,
    args=peft_training_args,
    train_dataset=tokenized_tiny_dataset["train"],
    data_collator=data_collator
)

peft_trainer.train()

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Step,Training Loss
1,2.279000
2,1.909300
3,2.184800
4,2.038900
5,2.284600


TrainOutput(global_step=5, training_loss=2.1393206119537354, metrics={'train_runtime': 39.595, 'train_samples_per_second': 1.01, 'train_steps_per_second': 0.126, 'total_flos': 3802367262720.0, 'train_loss': 2.1393206119537354, 'epoch': 1.67})

**3.3 — Evaluate the Model Qualitatively**

In [19]:
peft_demo_model.eval()

def generate_peft_summary(model, dialogue):
    prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""
    inputs = demo_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    output_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=80
    )

    return demo_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

In [20]:
peft_after_training_summaries = []

for dialogue in test_dialogues:
    peft_after_training_summaries.append(
        generate_peft_summary(peft_demo_model, dialogue)
    )

dash_line = "-".join("" for x in range(100))

for i in range(len(test_dialogues)):
    print(dash_line)
    print(f"Example {i + 1}")
    print(dash_line)
    print("HUMAN BASELINE SUMMARY:")
    print(human_summaries[i])
    print(dash_line)
    print("MODEL BEFORE MINI FINE-TUNING:")
    print(before_training_summaries[i])
    print(dash_line)
    print("FULL MINI FINE-TUNING MODEL:")
    print(after_training_summaries[i])
    print(dash_line)
    print("PEFT / LORA MODEL:")
    print(peft_after_training_summaries[i])
    print()

---------------------------------------------------------------------------------------------------
Example 1
---------------------------------------------------------------------------------------------------
HUMAN BASELINE SUMMARY:
#Person1# and #Person2# talk about pilgrims around the world, including Muslims' pilgrimage to mecca and Christians' pilgrimage to Canterbury or Vatican. #Person2# thinks faith heals people instead of magical places.
---------------------------------------------------------------------------------------------------
MODEL BEFORE MINI FINE-TUNING:
Muslims.
---------------------------------------------------------------------------------------------------
FULL MINI FINE-TUNING MODEL:
#Person1#: I am a pilgrim. #Person2#: I am a muslim. #Person1#: I am a Muslim. #Person2#: I am a muslim. #Person1#: #Person2#: I am a Muslim. #Person1#: #Person2
---------------------------------------------------------------------------------------------------
PEFT / LORA MODEL:

**Notes**

The qualitative evaluation shows that the PEFT/LoRA model changed the model behavior and often produced more summary-like outputs than the original baseline. In several examples, the PEFT model captured more context than the original model and produced more complete responses than the mini full fine-tuned model. However, the summaries were still imperfect because the experiment used a very small dataset and only 5 training steps. This confirms that PEFT can efficiently adapt a model, but meaningful performance still requires enough data and training time.

**3.4 — Evaluate the Model Quantitatively with ROUGE Metric**

In [21]:
rouge = evaluate.load("rouge")

peft_results = rouge.compute(
    predictions=peft_after_training_summaries,
    references=human_summaries,
    use_aggregator=True,
    use_stemmer=True
)

print("BEFORE MINI FINE-TUNING:")
print(before_results)

print("\nFULL MINI FINE-TUNING:")
print(after_results)

print("\nPEFT / LORA MODEL:")
print(peft_results)

BEFORE MINI FINE-TUNING:
{'rouge1': 0.10436432637571157, 'rouge2': 0.0, 'rougeL': 0.10436432637571158, 'rougeLsum': 0.10436432637571158}

FULL MINI FINE-TUNING:
{'rouge1': 0.2814872866597004, 'rouge2': 0.03125, 'rougeL': 0.24508881922675024, 'rougeLsum': 0.24508881922675022}

PEFT / LORA MODEL:
{'rouge1': 0.3198953823953824, 'rouge2': 0.03485576923076923, 'rougeL': 0.27065295815295815, 'rougeLsum': 0.27065295815295815}


In [22]:
print("Absolute improvement of PEFT over original baseline:")

improvement = (
    np.array(list(peft_results.values())) -
    np.array(list(before_results.values()))
)

for key, value in zip(peft_results.keys(), improvement):
    print(f"{key}: {value * 100:.2f}%")

Absolute improvement of PEFT over original baseline:
rouge1: 21.55%
rouge2: 3.49%
rougeL: 16.63%
rougeLsum: 16.63%


In [23]:
print("Absolute difference of PEFT compared to full mini fine-tuning:")

difference = (
    np.array(list(peft_results.values())) -
    np.array(list(after_results.values()))
)

for key, value in zip(peft_results.keys(), difference):
    print(f"{key}: {value * 100:.2f}%")

Absolute difference of PEFT compared to full mini fine-tuning:
rouge1: 3.84%
rouge2: 0.36%
rougeL: 2.56%
rougeLsum: 2.56%


**Conclusion Notes Lab 2**

**Fine-Tuning and PEFT/LoRA for Dialogue Summarization :**


* This lab demonstrated the full workflow of adapting a generative AI model for dialogue summarization. The original FLAN-T5 model was tested with zero-shot inference, then the dataset was preprocessed into input_ids and labels for training.

* Full fine-tuning was shown to be computationally expensive because it requires updating all model parameters. To understand the process in a lightweight way, a small FLAN-T5 model was fine-tuned on a tiny subset of DialogSum. The mini fine-tuning experiment changed the model outputs and improved ROUGE scores compared to the original baseline.

* The lab also demonstrated Parameter Efficient Fine-Tuning using LoRA. Instead of training the entire model, LoRA trained only 1.76% of the parameters. Despite using far fewer trainable parameters, the PEFT/LoRA model achieved better ROUGE scores than the original baseline and slightly better results than the mini full fine-tuning experiment.

* Overall, the lab shows that fine-tuning can improve task-specific performance, but full fine-tuning requires significant compute resources. PEFT/LoRA provides a more efficient alternative by adapting the model with fewer trainable parameters while still achieving meaningful performance improvements.